In [1]:
import pickle

with open('data/train_preprocessed2.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

d:\ComputerScience\BachKhoa\ProjectII\YOLOQA\venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_encodings.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])

In [3]:
from preprocess import SquadDataset
from torch.utils.data import DataLoader

train_dataset = SquadDataset(train_encodings)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)

In [5]:
item = next(iter(train_loader))


In [6]:
item.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])

In [7]:
print(type(item['input_ids']))
print(item['input_ids'].shape)

<class 'torch.Tensor'>
torch.Size([1, 512])


In [8]:
print(item['start_positions'])
print(item['end_positions'])

tensor([67])
tensor([70])


In [9]:
import torch
import json


class SquadDataset(torch.utils.data.Dataset):
    def __init__(self, file_path, tokenizer):
        with open(file_path, 'r') as f:
            self.data = json.load(f)

        self.tokenizer = tokenizer
        self.contexts = []
        self.questions = []
        self.answers = []
        for group in self.data['data']:
            for passage in group['paragraphs']:
                context = passage['context']
                for qa in passage['qas']:
                    question = qa['question']
                    for answer in qa['answers']:
                        self.answers.append(answer)
                        self.questions.append(question)
                        self.contexts.append(context)

    def convert_to_token_start_end_pos(self, encodings, answer):
        start = encodings.char_to_token(0, answer['answer_start'])
        end = encodings.char_to_token(0, answer['answer_end'])

        # if start = None, the answers have been truncated
        if start == None:
            start = self.tokenizer.model_max_length

        # if end == None, the 'char_to_token' function points to the space after the correct token, so add - 1
        if end == None:
            end = encodings.char_to_token(0, answer['answer_end'] - 1)
            # if end is still None, the answers have been truncated
            if end == None:
                end = self.tokenizer.model_max_length

        encodings['start_positions'] = start
        encodings['end_positions'] = end

        return encodings

    def __getitem__(self, index):
        context = self.contexts[index]
        question = self.questions[index]
        answer = self.answers[index]

        real_answer = answer['text']
        start_idx = answer['answer_start']
        # Get the real end index
        end_idx = start_idx + len(real_answer)

        # Deal with the problem of 1 or 2 more characters 
        if context[start_idx:end_idx] == real_answer:
            answer['answer_end'] = end_idx
        # When the real answer is more by one character
        elif context[start_idx-1:end_idx-1] == real_answer:
            answer['answer_start'] = start_idx - 1
            answer['answer_end'] = end_idx - 1  
        # When the real answer is more by two characters  
        elif context[start_idx-2:end_idx-2] == real_answer:
            answer['answer_start'] = start_idx - 2
            answer['answer_end'] = end_idx - 2

        encodings = self.tokenizer(context, question, truncation=True, padding=True)
        encodings = self.convert_to_token_start_end_pos(encodings, answer)

        return {key: torch.tensor(val) for key, val in encodings.items()}

    def __len__(self):
        return len(self.questions)

In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [11]:
train_data = SquadDataset('data/train-v2.0.json', tokenizer)

In [12]:
class CustomCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, samples):
        batch_size = len(samples)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'

        sample = samples[0]

        max_seq_length = tokenizer.model_max_length
        padded_length = max_seq_length

        input_shape = (1, padded_length)
        input_ids = torch.full(input_shape,
                               self.pad_token_id,
                               dtype=torch.long)
        attention_mask = torch.zeros(input_shape, dtype=torch.long)

        seq_length = len(sample['input_ids'])
        input_ids[0, :seq_length] = sample['input_ids']
        attention_mask[0, :seq_length] = sample['attention_mask']

        start_positions = sample['start_positions']
        end_positions = sample['end_positions']

        return dict(input_ids=input_ids,
                    attention_mask=attention_mask,
                    start_positions=start_positions,
                    end_positions=end_positions)

In [13]:
from torch.utils.data import DataLoader
collate = CustomCollator(tokenizer)
train_dl = DataLoader(train_data,
                      batch_size=1,
                      shuffle=False,
                      collate_fn=collate)

In [14]:
new_item = next(iter(train_dl))

In [15]:
print(item.keys())
print(new_item.keys())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])
dict_keys(['input_ids', 'attention_mask', 'start_positions', 'end_positions'])


In [17]:
(item['input_ids'] == new_item['input_ids']).all()

tensor(True)

In [18]:
(item['attention_mask'] == new_item['attention_mask']).all()

tensor(True)

In [19]:
item['start_positions'] == new_item['start_positions']

tensor([True])

In [20]:
item['end_positions'] == new_item['end_positions']

tensor([True])